# DAE Registration Quick Notebook

This notebook is the small public DAE example for `EOptInterface.jl`.

Its purpose is narrow and practical:
- show a tiny MTK system that stays a DAE after `structural_simplify(...)`;
- show how `register_daesystem(...)` is called;
- compare several supported time-stepping methods on the same DAE;
- save the results for later inspection.

This is not a full MPC notebook.
It is a registration-layer notebook.


In [ ]:
using Pkg

repo_root = isdir(joinpath(pwd(), "src")) ? pwd() : normpath(joinpath(pwd(), ".."))
examples_env = joinpath(repo_root, "examples")
examples_project = joinpath(examples_env, "Project.toml")
generated_dir = joinpath(repo_root, "examples", "generated")
style_path = joinpath(repo_root, "notebooks", "eoi_publication_plots.jl")

# Set this to true only when bootstrapping a fresh examples environment.
bootstrap_examples_env = false

if Base.active_project() != examples_project
    Pkg.activate(examples_env; io=devnull)
end

if bootstrap_examples_env || !isfile(joinpath(examples_env, "Manifest.toml"))
    Pkg.instantiate(; io=devnull)
end

using CSV, DataFrames, Plots

include(style_path)
apply_eoi_publication_style!()


## Re-run The DAE Example

This cell runs `examples/dae_registration_demo.jl`.

The model is intentionally small:
- `D(x) = -x + z + u`
- `0 = z + sin(z) - x`

The second equation is implicit.
That is the key feature.
It prevents the algebraic part from disappearing during simplification, so the example remains a real DAE test case.


In [ ]:
rerun_demo = true

example_script = joinpath(repo_root, "examples", "dae_registration_demo.jl")

if rerun_demo
    cmd = `$(Base.julia_cmd()) --project=$(examples_env) $example_script`
    run(cmd)
end


In [ ]:
structure_path = joinpath(generated_dir, "dae_registration_structure_summary.csv")
equations_path = joinpath(generated_dir, "dae_registration_equations.csv")
integrator_path = joinpath(generated_dir, "dae_registration_integrator_summary.csv")
trajectory_path = joinpath(generated_dir, "dae_registration_trajectories.csv")

for path in (structure_path, equations_path, integrator_path, trajectory_path)
    isfile(path) || error("Missing $(path). Run the example cell first.")
end

structure_summary = CSV.read(structure_path, DataFrame)
equation_rows = CSV.read(equations_path, DataFrame)
integrator_summary = CSV.read(integrator_path, DataFrame)
trajectories = CSV.read(trajectory_path, DataFrame)

structure_summary


## Structure Check

This table is the first thing to inspect.
It answers one simple question:
"Did the model stay a DAE after simplification?"

If the simplified system still has both:
- a differential equation;
- an algebraic equation;
then the test case is doing its job.


In [ ]:
equation_rows


In [ ]:
integrator_summary


In [ ]:
p_x = plot(title = "Differential state x(t)", xlabel = "Time", ylabel = "x")
p_z = plot(title = "Algebraic state z(t)", xlabel = "Time", ylabel = "z")
p_u = plot(title = "Discretized input u[k]", xlabel = "Time", ylabel = "u")

for integrator in unique(trajectories.integrator)
    sub = trajectories[trajectories.integrator .== integrator, :]
    plot!(p_x, sub.time, sub.x, label = integrator)
    plot!(p_z, sub.time, sub.z, label = integrator)
    plot!(p_u, sub.time, sub.u, label = integrator, seriestype = :steppost)
end

plot(p_x, p_z, p_u, layout = (3, 1), size = (980, 980))


dae_plot_registration_triplet(trajectories)
